In [2]:
import zipfile
from google.cloud import storage
import os
import pandas as pd
import datetime as dt
import geopandas as gpd
import shapely
import pyproj
import numpy as np
import matplotlib.pyplot as plt
from OSGridConverter import grid2latlong
from fuzzywuzzy import fuzz, process

## WWTW - TO CHECK

In [ ]:
determinands = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/Determinands.csv"
)

In [ ]:
c_active = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/consents_active.csv",
    encoding="latin-1"
)

In [ ]:
wwtw = c_active[c_active["DISCHARGE_SITE_TYPE_CODE"] == 'A2']

In [ ]:
wwtw.columns, determinands.columns

In [ ]:
dete_codes = [7782, 7729, 135, 85, 111, 6051, 348, 6057, 3647, 8174, 6425]

In [ ]:
determinands = determinands[determinands["DETE_CODE"].isin(dete_codes)]

In [ ]:
# Get code values together

determinands['CODE_VALUE'] = determinands.apply(
    lambda x: [
        (x[f'CODE_{i}'], x[f'VAL_{i}'])
        for i in range(1, 4)
        if pd.notna(x[f'CODE_{i}']) and pd.notna(x[f'VAL_{i}'])
    ],
    axis=1
)
determinands = determinands.explode('CODE_VALUE')
determinands[['CODE', 'VALUE']] = pd.DataFrame(
    determinands['CODE_VALUE'].tolist(), index=determinands.index)
determinands = determinands.drop(['CODE_VALUE', 'CODE_1', 'CODE_2', 'CODE_3',
                                  'VAL_1', 'VAL_2', 'VAL_3'], axis=1)

In [ ]:
determinands

In [ ]:
pivoted_determinands = pd.pivot_table(determinands, values="VALUE", index=[
                                      "PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER", "CODE"], columns=["DETE"])

In [ ]:
wwtw_ren = wwtw.rename(
    columns={"PERMIT_NUMBER": "PERMIT_REF", "PERMIT_VERSION": "VERSION"})
wwtw_ren.set_index(
    ["PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER"])

In [ ]:
wwtw_cleaned = wwtw_ren.set_index(["PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER"]).join(
    pivoted_determinands, how='left')

In [ ]:
gdf = gpd.GeoDataFrame(wwtw_cleaned)

In [ ]:
gdf

In [ ]:
gdf["OUTLET_LOC"] = gdf["OUTLET_GRID_REF"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_GRID_REF"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_NGR"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))

In [ ]:
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_LOC"].astype('geometry')
gdf["OUTLET_LOC"] = gdf["OUTLET_LOC"].astype('geometry')
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_LOC"].astype('geometry')

In [ ]:
gdf = gdf.set_geometry("DISCHARGE_LOC").set_crs(epsg=4326)

In [ ]:
gdf = gdf.reset_index()

In [ ]:
gdf.shape

In [ ]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/WWTW.parquet")

## SSSIs

In [144]:
bucketName = "gs://rdmai_dev_data/raw/CSO_Historical_Data/NWG/".split("/")[2]
path_in_bucket = '/'.join(
    "gs://rdmai_dev_data/raw/CSO_Historical_Data/NWG/".split("/")[3:])
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]
cso = gpd.read_file(geojson_filenames[-1])
cso_all = cso.to_crs(epsg=27700)

In [145]:
gdf = gpd.read_file(
    "gs://rdmai_dev_data/raw/Sites_of_Special_Scientific_Interest/Sites_of_Special_Scientific_Interest_England.zip")

In [146]:
gdf = gdf.to_crs(epsg=27700)

In [147]:
gdf.to_file("gs://rdmai_dev_data/cleansed/Theme2CD/SSSI.geojson")

In [ ]:
gpd.read_file("gs://rdmai_dev_data/cleansed/Theme2CD/SSSI.geojson")

## Bathing Waters

In [148]:
distance_joined = pd.read_csv("gs://rdmai_dev_data/raw/Bathing_Water_Data.csv")

In [150]:
gdf = gpd.GeoDataFrame(distance_joined, geometry=gpd.points_from_xy(
    distance_joined["long"], distance_joined["lat"]), crs="EPSG:4326").to_crs(epsg=27700)

In [ ]:
gdf.columns

In [151]:
gdf.to_file("gs://rdmai_dev_data/cleansed/Theme2CD/BathingWaters.geojson")

## Consents

In [53]:
def get_lat_lon(x):
    try:
        latlong = grid2latlong(x)
        return shapely.Point(latlong.longitude, latlong.latitude)
    except:
        return None

In [54]:
gdf = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/consents_active.csv",
    encoding="latin-1"
)

### WWTW

In [173]:
rec_env_codes = [
    1,  # Freshwater River
    6,  # Canal
    # 7,  # Pond/Lake/Reservoir
    9,  # Estuary
    15,  # Into land then to watercourse
]

In [174]:
gdf = gdf[
    (gdf["DISCHARGE_SITE_TYPE_CODE"] == "A2") &
    (gdf["EFFLUENT_TYPE"] == "SA") &
    ((gdf["OUTLET_TYPE_CODE"] == "S") | (gdf["OUTLET_TYPE_CODE"] == "U")) &
    (gdf["RECEIVING_ENVIRON_TYPE_CODE"].isin(rec_env_codes))
]

In [175]:
gdf = gdf.drop_duplicates(
    subset=["PERMIT_NUMBER", "OUTLET_GRID_REF"], keep='first')

In [176]:
gdf["ID"] = gdf.apply(lambda x: str(x["PERMIT_NUMBER"]+"_" +
                      str(x["EFFLUENT_NUMBER"]) + "_" + str(x["OUTLET_NUMBER"])), axis=1)
gdf.set_index("ID", inplace=True)

In [177]:
gdf["OUTLET_LOC"] = gdf["OUTLET_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_NGR"].apply(
    get_lat_lon).astype('geometry')

In [178]:
gdf = gdf.set_geometry("OUTLET_LOC").set_crs(epsg=4326).to_crs(epsg=27700)
gdf = gdf.set_geometry("EFFLUENT_LOC").set_crs(epsg=4326).to_crs(epsg=27700)
gdf = gdf.set_geometry("DISCHARGE_LOC").set_crs(epsg=4326).to_crs(epsg=27700)

In [179]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")

In [ ]:
gdf = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")

### CSOs

In [49]:
# rec_env_codes = [
#     1,  # Freshwater River
#     6,  # Canal
#     # 7,  # Pond/Lake/Reservoir
#     9,  # Estuary
#     15,  # Into land then to watercourse
# ]
effluent_types = ["SB", "SC", "SD"]

In [50]:
gdf = gdf[
    # (gdf["OUTLET_TYPE_CODE"] == "S") &
    # (gdf["RECEIVING_ENVIRON_TYPE_CODE"].isin(rec_env_codes)) &
    (gdf["EFFLUENT_TYPE"].isin(effluent_types))
]

In [51]:
gdf = gdf.drop_duplicates(
    subset=["PERMIT_NUMBER", "OUTLET_GRID_REF"], keep='first')

In [52]:
gdf["ID"] = gdf.apply(lambda x: str(x["PERMIT_NUMBER"]+"_" +
                      str(x["EFFLUENT_NUMBER"]) + "_" + str(x["OUTLET_NUMBER"])), axis=1)

In [8]:
gdf["OUTLET_LOC"] = gdf["OUTLET_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_NGR"].apply(
    get_lat_lon).astype('geometry')

In [9]:
gdf = gdf.set_geometry("OUTLET_LOC").set_crs(epsg=4326).to_crs(epsg=27700)
gdf = gdf.set_geometry("EFFLUENT_LOC").set_crs(epsg=4326).to_crs(epsg=27700)
gdf = gdf.set_geometry("DISCHARGE_LOC").set_crs(epsg=4326).to_crs(epsg=27700)

In [10]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

In [ ]:
gdf = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

## Determinands

In [180]:
determinands = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/Determinands.csv"
)

/var/tmp/ipykernel_805272/3571112617.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  determinands = pd.read_csv(


In [181]:
dete_codes = [
    7782,                   # Flow : Dry Weather :- {DWF}
    7729,                   # Flow : Dry Weather : 3 in 5 Years
    135,                    # Solids, Suspended at 105 C
    85,                     # BOD : 5 Day ATU
    111,                    # Ammoniacal Nitrogen as N
    6051,                   # Iron
    348,                    # Phosphorus, Total as P
    6057,                   # Aluminium
    3647,                   # Flow : To full treatment (Before spill to stor...
    8174,                   # Weir Setting
    6425,                   # Flow : Daily total
]

In [182]:
determinands = determinands[determinands["DETE_CODE"].isin(dete_codes)]

In [183]:
# Get code values together

determinands['CODE_VALUE'] = determinands.apply(
    lambda x: [
        (x[f'CODE_{i}'], x[f'VAL_{i}'])
        for i in range(1, 4)
        if pd.notna(x[f'CODE_{i}']) and pd.notna(x[f'VAL_{i}'])
    ],
    axis=1
)
determinands = determinands.explode('CODE_VALUE')
determinands[['CODE', 'VALUE']] = pd.DataFrame(
    determinands['CODE_VALUE'].tolist(), index=determinands.index)
determinands = determinands.drop(['CODE_VALUE', 'CODE_1', 'CODE_2', 'CODE_3',
                                  'VAL_1', 'VAL_2', 'VAL_3'], axis=1)

In [184]:
determinands = determinands.reset_index(drop=True)

In [185]:
determinands.columns

Index(['EA_REGION', 'REGION', 'PERMIT_REF', 'VERSION', 'OUTLET_NUMBER',
       'EFFLUENT_NUMBER', 'MONTH_FROM', 'MONTH_TO', 'DETE_CODE', 'UNITS',
       'DETE', 'METHOD', 'CODE', 'VALUE'],
      dtype='object')

In [188]:
versionMask = determinands.groupby(["PERMIT_REF","OUTLET_NUMBER","EFFLUENT_NUMBER","DETE","CODE"])["VERSION"].idxmax()
determinands = determinands.loc[versionMask].reset_index(drop=True)

In [191]:
determinands.set_index(determinands.apply(
    lambda x: str(x["PERMIT_REF"]+"_" +
                  str(x["EFFLUENT_NUMBER"]) + "_" + str(x["OUTLET_NUMBER"])), axis=1),inplace=True)

In [192]:
determinands.to_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")

## Risk calculation

In [11]:
def get_lat_lon(x):
    try:
        latlong = grid2latlong(x)
        return shapely.Point(latlong.longitude, latlong.latitude)
    except:
        return None

### CSOs

In [41]:
# csos = gpd.read_file(
#     "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_arcgis.geojson")
df = pd.read_excel(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_Annual_2024.xlsx", header=1, sheet_name=None, index_col=None)
annual = pd.concat(df.values())

In [13]:
consents = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

In [14]:
determinands = pd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")

In [15]:
annual["Outlet Discharge NGR\n(EA Consents Database)"] = annual["Outlet Discharge NGR\n(EA Consents Database)"].apply(
    lambda x: x.split(" ")[0].split(",")[0])

In [16]:
annual["geometry"] = annual["Outlet Discharge NGR\n(EA Consents Database)"].apply(
    get_lat_lon).astype('geometry')

In [17]:
annual = gpd.GeoDataFrame(annual, geometry="geometry",
                          crs="EPSG:4326").to_crs(epsg=27700)

In [18]:
distance_joined = gpd.sjoin(
    annual.set_geometry(annual.buffer(5000).rename("buffer")),
    consents,
    predicate='contains',
    how="left",
)
distance_joined.set_geometry("geometry").drop("buffer",axis=1, inplace=True)

In [19]:
distance_joined["EA Permit Reference\n(EA Consents Database)"] = distance_joined["EA Permit Reference\n(EA Consents Database)"].astype(
    "str")
distance_joined["WaSC Supplementary Permit Ref.\n[optional]"] = distance_joined["WaSC Supplementary Permit Ref.\n[optional]"].astype(
    "str")
distance_joined["PERMIT_NUMBER"] = distance_joined["PERMIT_NUMBER"].astype(
    "str")

In [20]:
distance_joined["score"] = distance_joined.apply(
    lambda x: max(fuzz.ratio(x["EA Permit Reference\n(EA Consents Database)"],
                             x["PERMIT_NUMBER"]),
                  fuzz.ratio(x["WaSC Supplementary Permit Ref.\n[optional]"],
                             x["PERMIT_NUMBER"])),
    axis=1
)

In [21]:
distance_joined.reset_index(drop=True, inplace=True)
distance_joined.loc[distance_joined["score"] <= 92, "ID"] = np.nan
# Keep the max score for each unique ID
mask = distance_joined['score'] == distance_joined.groupby('Unique ID')[
    'score'].transform('max')
filtered = distance_joined[mask]

In [22]:
filtered = filtered[~((filtered["ID"].isna()) & (
    filtered.duplicated("Unique ID")))]

In [23]:
filtered["distance"] = filtered.apply(
    lambda x: x["OUTLET_LOC"].distance(x["geometry"]) if x["OUTLET_LOC"] is not None else None, axis=1)
dmask = filtered["distance"] == filtered.groupby(
    "Unique ID")["distance"].transform("min")
filtered = filtered[dmask]

In [24]:
cut = filtered[["Unique ID", "ID", "EA Permit Reference\n(EA Consents Database)",
                "WaSC Supplementary Permit Ref.\n[optional]", "Total Duration (hh:mm:ss) all spills prior to processing through 12-24h count method", "PERMIT_NUMBER", "score", "geometry", "OUTLET_LOC","distance"]]
cut.rename(columns={
           "Total Duration (hh:mm:ss) all spills prior to processing through 12-24h count method":"Spill Duration"}, inplace = True)

/var/tmp/ipykernel_860407/4219160323.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cut.rename(columns={


In [66]:
joined = cut.join(determinands[(determinands["DETE_CODE"]==8174) & (determinands["CODE"]=="MINIMUM VALUE")]["VALUE"], how="left", on = "ID").rename(columns={"VALUE":"Weir Setting"})

In [67]:
joined.set_index("Unique ID", inplace = True, verify_integrity=True)

In [35]:
a = gpd.GeoDataFrame(joined, geometry="geometry")

In [36]:
a.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/CSO_Annual_2024_Weir.parquet")

In [4]:
df = gpd.read_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/CSO_Annual_2024_Weir.parquet")
df

,ID,EA Permit Reference\n(EA Consents Database),WaSC Supplementary Permit Ref.\n[optional],Spill Duration,PERMIT_NUMBER,score,geometry,OUTLET_LOC,distance,Weir Setting
Unique ID,,,,,,,,,,
AWS00001,AWNNF13860_1_1,AWNNF13860,AWNNF4102,0 days 04:30:00,AWNNF13860,100,POINT (490382.467 274740.557),POINT (490382.467 274740.557),0.000000,NaN
AWS00002,AWNNF13859_1_1,AWNNF13859,AWNNF4103,0 days 05:30:00,AWNNF13859,100,POINT (490382.467 274740.557),POINT (490382.467 274740.557),0.000000,NaN
AWS00004,None,AW5NF311,AW5NF311(4)A,21 days 11:00:00,AW5NF1771,82,POINT (490652.532 268000.558),POINT (494714.29 267570.602),4084.450351,NaN
AWS00005,AWNNF13858_1_1,AWNNF13858,AWNNF4104/13858,0 days 01:30:00,AWNNF13858,100,POINT (490382.467 274740.557),POINT (490382.467 274740.557),0.000000,NaN
AWS00006,AWNNF13856_1_1,AWNNF13856,AWNNF4108,2 days 15:30:00,AWNNF13856,100,POINT (490302.354 263030.554),POINT (490302.354 263030.554),0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...
YWS02291,None,Permit applied for,E807(SS),4 days 14:15:00,25/02/1087,22,POINT (411232.028 509943.307),POINT (410883.025 514547.307),4617.208526,NaN
YWS02293,WRA7536_1_1,WRA7536,WRA7536,2 days 18:30:00,WRA7536,100,POINT (442263.388 427568.328),POINT (442263.388 427568.328),0.000000,118.0
YWS02294,27/27/0096_1_1,Permit applied for,27/27/0096,5 days 11:30:00,27/27/0096,100,POINT (511861.494 478563.954),POINT (511876.504 478560.954),15.307401,NaN


In [6]:
df.loc["NWL00328"]

ID                                                                            244/0921_1_1
EA Permit Reference\n(EA Consents Database)                                       244/0921
WaSC Supplementary Permit Ref.\n[optional]                                        244/0921
Spill Duration                                                             0 days 00:00:00
PERMIT_NUMBER                                                                     244/0921
score                                                                                  100
geometry                                       POINT (417230.1005925991 543100.3046313742)
OUTLET_LOC                                     POINT (417230.1005925991 543100.3046313742)
distance                                                                               0.0
Weir Setting                                                                           NaN
Name: NWL00328, dtype: object

### WwTW

In [ ]:
wwtw = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")
determinands = pd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")

#### Phosphorus

In [ ]:
joined = wwtw.join(determinands[determinands["DETE"].str.contains("Phosphorus") & (
    determinands["CODE"] == "MEAN VALUE")].set_index("ID")["VALUE"], on="ID", how="left")
joined.rename(columns={"VALUE": "P"}, inplace=True)

In [ ]:
joined["P"].fillna(5, inplace=True)

#### DWF

In [ ]:
joined = joined.join(determinands[(determinands["DETE"].str.contains("Flow : Dry Weather :- {DWF}")) & (
    determinands["CODE"] == "MAXIMUM VALUE")].set_index("ID")["VALUE"], on="ID", how="left")
joined.rename(columns={"VALUE": "DWF"}, inplace=True)

In [ ]:
joined["DWF"].fillna(25, inplace=True)

## Testing

In [ ]:
determinands

In [ ]:
determinands.set_index("ID", inplace=True)

In [ ]:
determinands[determinands["CODE"] ==
             "MAXIMUM VALUE"].loc["AWCNF10454_1_1_1", "VALUE"]

In [ ]:
determinands.to_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")